# 20 — DaeBuilder, FMI, and the plugin map

**Objectives.** Know when to reach for model-exchange tooling, discover installed plugins, and distinguish portable core examples from optional integrations.

`DaeBuilder` organizes time, states, derivatives, algebraic variables, controls, parameters, outputs, equations, and events. CasADi 3.7 can import FMI 2/3 FMUs through its native interface. FMU behavior depends on the archive and platform, so this portable course explains the workflow without committing a third-party binary.


## Learning path and assumptions

**Prerequisites:** Chapters 07–08 and 19.

**Problem:** Build a named dynamical model and inspect available numerical plugins.

**Assumptions and conventions:** Core lessons run without external FMU files or licensed plugins.

**How to read the results:** Installed capability, available derivatives, and suitability for a problem are separate checks.

### Resolve the local checkout

This setup locates repository-local presentation and numerical helpers and enables autoreload. It changes Python import resolution, not the mathematical model. Run it once after restarting the kernel.

In [ ]:
# Make repository-local helpers importable without installing a package.
import sys
from pathlib import Path

repository_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

# Autoreload picks up edits to tutorial models before each notebook cell runs.
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

### Inspect model-building and plugin capabilities

`ca.DaeBuilder` provides an incremental, named representation for
differential-algebraic models and imported model-exchange workflows. An
empty builder is sufficient to expose its interface here. The cell also
parses `CasadiMeta.plugins()` to group the solver and numerical backends
compiled into the active CasADi installation.

A general semi-explicit DAE represented by a populated builder has the form

$$
\dot x=f(t,x,z,u,p),\qquad
0=g(t,x,z,u,p),\qquad
y=h(t,x,z,u,p).
$$

Plugin families provide algorithms that operate on related mathematical
objects: `Integrator` advances ODEs/DAEs, `Rootfinder` solves
$g(z,p)=0$, `Nlpsol` solves nonlinear programs, `Conic` handles
quadratic/conic problems, `Linsol` solves linear systems, and `Interpolant`
constructs lookup-table functions. The printed list is capability
discovery, not a guarantee that every plugin is appropriate for every
model.


In [ ]:
import casadi as ca

# DaeBuilder incrementally describes named differential-algebraic models.
# This empty instance exposes the API without assuming an FMU workflow.
builder = ca.DaeBuilder("empty_teaching_model")
print(builder)

# CasadiMeta.plugins returns semicolon-separated Family::plugin entries
# for every backend compiled into the active CasADi distribution.
installed_plugins = ca.CasadiMeta.plugins().split(";")
plugins_by_family = {}
for plugin_entry in installed_plugins:
    family, separator, plugin_name = plugin_entry.partition("::")
    if separator:
        plugins_by_family.setdefault(family, []).append(plugin_name)

plugin_families = [
    "Nlpsol",
    "Conic",
    "Integrator",
    "Rootfinder",
    "Linsol",
    "Interpolant",
]
for family in plugin_families:
    print(family, sorted(plugins_by_family.get(family, [])))

## FMU workflow

1. Validate the FMU independently and record FMI version/platform support.
2. Construct/import a `DaeBuilder`, inspect variable categories, units, causality, equations, and events.
3. Create selected input/output/Jacobian functions with the function factory.
4. Compare nominal outputs and event behavior against the source model.
5. Check which generated functions support differentiation and code generation before embedding them in an NLP.

## Specialized functionality map

Use `nlpsol`, `conic`, `integrator`, `rootfinder`, `linsol`, and `interpolant` plugin registries rather than assuming a plugin ships in every wheel. Proprietary solvers need their own installation/licenses. Lower-level sparsity ordering/coloring, custom `Callback` derivatives, external functions, serialization, importers, and solver-specific options belong in the API reference once the core workflow is understood.

**Recap.** Discover capabilities at runtime, keep optional integrations isolated, and test imported models numerically before differentiating or optimizing them.


## Populate and evaluate a DaeBuilder model

Create named scalar state $x$ and tunable parameter $p$, then register $\dot x=-px$. `der(x)` identifies a derivative variable; `eq` attaches its equation. Tunable variability is explicit so the parameter remains a runtime input rather than a fixed constant. The factory exposes an ordinary Function with input groups `x`, `p` and output `ode`.

In [ ]:
import numpy as np
from casadi_tutorial.plotting import plot_series, display_plot
populated=ca.DaeBuilder("named_decay")
named_state=populated.add("x",{"start":1.0})
named_rate=populated.add("p","parameter","tunable",{"start":2.0})
populated.eq(populated.der(named_state),-named_rate*named_state)
named_rhs=populated.create("named_rhs",["x","p"],["ode"])
assert abs(float(named_rhs(1.0,2.0))+2.0)<1e-12
print(populated)
print("State names:",populated.x(),"parameter names:",populated.p())

def builder_experiment(rate=2.0):
    rhs_value=float(named_rhs(1.0,rate))
    print("Factory evaluation f(1,p):",rhs_value)
    times=np.linspace(0,2,80)
    display_plot(plot_series(times,{"exact named model":np.exp(-rate*times)},title="Named decay model",ylabel="state x"))
builder_experiment()

**Explain:** the symbolic model is portable; an imported FMU still requires separate checks for platform compatibility, events, derivatives, and code-generation support. See the [CasADi DaeBuilder guide](https://web.casadi.org/docs/#the-daebuilder-class).

### Run your own experiment

The default above is already computed. Change the controls, predict the result, and press **Run experiment**. **Reset controls** restores the initial settings. Solver calls are explicit; dragging controls does not launch optimizations. Python controls require a live JupyterLab kernel.

In [ ]:
from casadi_tutorial.interactive import experiment
try:
    import ipywidgets as widgets
except ImportError:
    print("Install the course environment to enable live controls.")
else:
    experiment(builder_experiment, dict(rate=widgets.FloatSlider(value=2,min=.1,max=4,step=.1)), title='DaeBuilder parameter exploration')

## Recap and independent practice

State the mathematical object constructed in this module, the numerical assumptions it requires, and one check that could reveal a wrong result. Change one experimental setting at a time and explain the observed change before continuing.